# 第 10 章 ニューラルネットワーク

直線では分けられない XOR を、隠れ層を積むことで解きます。学習は誤差逆伝播法で行います。

対応する記事: [第 10 章 ニューラルネットワーク（Kotlin Notebook の言語版）](../../../docs/article/grokking-machine-learning/kotlin/ch10.md)

実装本体: `apps/grokking-ml-kotlin/src/`

## セットアップ

実装本体をビルドした JAR を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

先に JAR を作っておいてください。

```bash
cd apps/grokking-ml-kotlin
./gradlew jar
```

IntelliJ IDEA の Kotlin Notebook プラグイン、または [Kotlin Jupyter カーネル](https://github.com/Kotlin/kotlin-jupyter) で開きます。

```bash
pip install kotlin-jupyter-kernel
jupyter lab notebooks/
```

In [1]:
@file:DependsOn("../build/libs/grokking-ml-kotlin-0.1.0.jar")

import ch10.*

## XOR は直線で分けられない

対角線上の 2 点が同じクラスなので、**1 本の直線では絶対に分けられません。**

In [2]:
val points = listOf(listOf(0.0, 0.0), listOf(0.0, 1.0), listOf(1.0, 0.0), listOf(1.0, 1.0))
val labels = listOf(0, 1, 1, 0)

points.zip(labels).forEach { (point, label) ->
    println("(%.0f, %.0f) → %d".format(point[0], point[1], label))
}

(0, 0) → 0
(0, 1) → 1
(1, 0) → 1
(1, 1) → 0


## 隠れ層の幅を変えて比べる

**隠れ層があっても、ニューロンが 1 つでは足りません。** 実質「直線を 1 本引いてから変換する」だけなので、表現力はロジスティック回帰と変わらないからです。

**「層を足せば強くなる」ではなく「十分な幅の隠れ層が要る」** ということです。

In [3]:
listOf(1, 2, 4).forEach { hidden ->
    val (m, losses) = train(points, labels, hiddenSize = hidden, epochs = 20000, seed = 0)
    println("隠れ層 %d ニューロン  正解率 %.2f  損失 %.4f → %.4f".format(hidden,
            accuracy(m, points, labels), losses.first(), losses.last()))
}

隠れ層 1 ニューロン  正解率 0.75  損失 0.8160 → 0.5086


隠れ層 2 ニューロン  正解率 1.00  損失 0.8162 → 0.0010


隠れ層 4 ニューロン  正解率 1.00  損失 0.9238 → 0.0008


## 学習後の予測

隠れ層 4 ニューロンなら、**4 点すべてを 0.999 以上の確信で当てられます。**

In [4]:
val (model, losses) = train(points, labels, hiddenSize = 4, epochs = 20000, seed = 0)

points.zip(labels).forEach { (point, label) ->
    println("(%.0f, %.0f) 正解=%d  予測確率 %.4f".format(point[0], point[1], label,
            model.predictProbability(point)))
}

(0, 0) 正解=0  予測確率 0.0002
(0, 1) 正解=1  予測確率 0.9995


(1, 0) 正解=1  予測確率 0.9991
(1, 1) 正解=0  予測確率 0.0016


## 勾配消失

シグモイドの微分は最大 0.25、両端では 0 に近づきます。**層を深く積むとこの小さな値が掛け合わされ、入力側の層がほとんど学習しなくなります。**

現代のネットワークが ReLU を使う理由がここにあります。

In [5]:
println("%8s %10s".format("出力", "微分"))
listOf(0.001, 0.1, 0.5, 0.9, 0.999).forEach { output ->
    println("%8.3f %10.6f".format(output, sigmoidDerivative(output)))
}

println()
println("10 層積んだときの積 %.2e".format(Math.pow(0.25, 10.0)))

      出力         微分
   0.001   0.000999
   0.100   0.090000
   0.500   0.250000


   0.900   0.090000
   0.999   0.000999

10 層積んだときの積 9.54e-07


## 試してみる: 学習の途中経過

損失がどう下がるかを見ます。**XOR は最初しばらく停滞してから、あるところで急に解けます。**

In [6]:
for (epoch in 0 until 20000 step 2000) {
    val bar = "#".repeat((losses[epoch] * 50).toInt())
    println("epoch %6d  損失 %.4f  %s".format(epoch, losses[epoch], bar))
}

epoch      0  損失 0.9238  ##############################################


epoch   2000  損失 0.0161  
epoch   4000  損失 0.0054  
epoch   6000  損失 0.0032  


epoch   8000  損失 0.0022  
epoch  10000  損失 0.0017  
epoch  12000  損失 0.0014  


epoch  14000  損失 0.0012  
epoch  16000  損失 0.0010  
epoch  18000  損失 0.0009  
